## Extração de Features — Rally

Calcula o percentual de vitórias por comprimento de rally (1-3, 4-6, 7+) em duas perspectivas: saque e retorno.
Ao final, salva `features_rally.csv` com as médias acumuladas pré-partida de cada jogador.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

rally = pd.read_csv(f'{BASE}/charting-m-stats-Rally.csv')

print('rally:', rally.shape)
rally.head(4)

rally: (96706, 13)


,match_id,server,returner,row,pts,pl1_won,pl1_winners,pl1_forced,pl1_unforced,pl2_won,pl2_winners,pl2_forced,pl2_unforced
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,Total,141,64,24,19,21,77,28,24,19
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,1-3,72,37,10,16,11,35,10,10,9
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,4-6,30,17,9,2,2,13,3,8,6
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,7-9,25,7,4,1,4,18,8,6,2


#### Filtrando por comprimento de rally e agrupando 7-9 e 10+ em '7plus'

In [2]:
rally_filt = rally[rally['row'].isin(['1-3', '4-6', '7-9', '10'])].copy()

rally_filt['Date'] = pd.to_datetime(
    rally_filt['match_id'].str.split('-').str[0], format='%Y%m%d', errors='coerce'
)

rally_filt['row'] = rally_filt['row'].replace({'7-9': '7plus', '10': '7plus'})

print(rally_filt['row'].value_counts())
print(rally_filt.shape)

row
7plus    14922
1-3       7558
4-6       7558
Name: count, dtype: int64
(30038, 14)


#### Agregando pontos por partida, jogador e comprimento de rally

In [3]:
rally_agg = rally_filt.groupby(['match_id', 'server', 'returner', 'row', 'Date'], as_index=False)[
    ['pts', 'pl1_won', 'pl2_won']
].sum()

rally_agg['serve_win_pct']  = rally_agg['pl1_won'] / rally_agg['pts'].replace(0, np.nan)
rally_agg['return_win_pct'] = rally_agg['pl2_won'] / rally_agg['pts'].replace(0, np.nan)

print(rally_agg['row'].value_counts())
print(rally_agg.shape)

row
1-3      7545
4-6      7545
7plus    7538
Name: count, dtype: int64
(22628, 10)


#### Separando perspectiva de saque e retorno

In [4]:
serve_df = (rally_agg[['match_id', 'server', 'row', 'Date', 'serve_win_pct']]
    .rename(columns={'server': 'player'})
    .pivot_table(index=['match_id', 'player', 'Date'], columns='row', values='serve_win_pct')
    .reset_index()
)
serve_df.columns = ['match_id', 'player', 'Date', 'srv_win_1_3', 'srv_win_4_6', 'srv_win_7plus']

return_df = (rally_agg[['match_id', 'returner', 'row', 'Date', 'return_win_pct']]
    .rename(columns={'returner': 'player'})
    .pivot_table(index=['match_id', 'player', 'Date'], columns='row', values='return_win_pct')
    .reset_index()
)
return_df.columns = ['match_id', 'player', 'Date', 'ret_win_1_3', 'ret_win_4_6', 'ret_win_7plus']

rally_by_player = serve_df.merge(return_df, on=['match_id', 'player', 'Date'], how='outer')

print(rally_by_player.shape)
rally_by_player.head(3)

(15090, 9)


,match_id,player,Date,srv_win_1_3,srv_win_4_6,srv_win_7plus,ret_win_1_3,ret_win_4_6,ret_win_7plus
0,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Luis Ayala,1960-05-29,NaN,NaN,NaN,0.459016,0.510204,0.517241
1,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Nicola Pietrangeli,1960-05-29,0.540984,0.489796,0.482759,NaN,NaN,NaN
2,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Neale Fraser,1960-07-04,NaN,NaN,NaN,0.523179,0.540541,0.500000


#### Calculando médias acumuladas pré-partida

In [5]:
rally_by_player = rally_by_player.sort_values(['player', 'Date'])

rally_cols = ['srv_win_1_3', 'srv_win_4_6', 'srv_win_7plus',
              'ret_win_1_3', 'ret_win_4_6', 'ret_win_7plus']

for col in rally_cols:
    rally_by_player[f'avg_{col}'] = (
        rally_by_player.groupby('player')[col].transform(lambda x: x.expanding().mean().shift(1))
    )

print(rally_by_player.shape)
rally_by_player.head(4)

(15090, 15)


,match_id,player,Date,srv_win_1_3,srv_win_4_6,srv_win_7plus,ret_win_1_3,ret_win_4_6,ret_win_7plus,avg_srv_win_1_3,avg_srv_win_4_6,avg_srv_win_7plus,avg_ret_win_1_3,avg_ret_win_4_6,avg_ret_win_7plus
484,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,1989-09-09,NaN,NaN,NaN,0.447619,0.44898,0.528302,NaN,NaN,NaN,NaN,NaN,NaN
532,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,1990-04-15,0.439024,0.454545,0.542857,NaN,NaN,NaN,NaN,NaN,NaN,0.447619,0.44898,0.528302
764,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,1991-09-02,0.567376,0.395833,0.540741,NaN,NaN,NaN,0.439024,0.454545,0.542857,0.447619,0.44898,0.528302
788,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,1991-10-26,0.413793,0.173913,0.538462,NaN,NaN,NaN,0.503200,0.425189,0.541799,0.447619,0.44898,0.528302


#### Selecionando apenas as colunas finais (avg_*)

In [6]:
meta_cols    = ['match_id', 'player', 'Date']
feature_cols = [f'avg_{c}' for c in rally_cols]

features_rally = rally_by_player[meta_cols + feature_cols].copy()

print(features_rally.shape)
print(f'NaN na primeira partida de cada jogador (esperado): {features_rally["avg_srv_win_1_3"].isna().sum()}')
features_rally.head(4)

(15090, 9)
NaN na primeira partida de cada jogador (esperado): 1613


,match_id,player,Date,avg_srv_win_1_3,avg_srv_win_4_6,avg_srv_win_7plus,avg_ret_win_1_3,avg_ret_win_4_6,avg_ret_win_7plus
484,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,1989-09-09,NaN,NaN,NaN,NaN,NaN,NaN
532,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,1990-04-15,NaN,NaN,NaN,0.447619,0.44898,0.528302
764,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,1991-09-02,0.439024,0.454545,0.542857,0.447619,0.44898,0.528302
788,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,1991-10-26,0.503200,0.425189,0.541799,0.447619,0.44898,0.528302


#### Salvando

In [7]:
features_rally.to_csv(f'{BASE}/features_rally.csv', index=False)
print('Salvo:', f'{BASE}/features_rally.csv')
print('Shape:', features_rally.shape)

Salvo: ../../new-dataset/tennis_MatchChartingProject-master/features_rally.csv
Shape: (15090, 9)
